# Caso F · 01 MLflow + lakeFS — visión general

> _Tutorial · Caso de uso: **F — MLOps** · Capa Medallion: **transversal** · Spec: `docs/specs/synthetic-bms/01-product-spec.md`_

Material docente del proyecto **CAPTIA Synthetic Data BMS** — IES Dr. Lluís Simarro,
Curso de Especialización IA & Big Data 2025-2026.


## 1. Objetivo

Entender los conceptos de experiment, run, artefacto y tag de dataset, y ver cómo MLflow + lakeFS resuelven la reproducibilidad sin reinventar la rueda.


## 2. Qué se aprende

- Diferencia entre experiment, run y artefacto.
- Cómo lakeFS versiona datasets como Git versiona código.
- Convención `experiment-name = caso-modelo-fecha`.


## 3. Contexto del caso de uso

El equipo F es transversal: define la convención que usarán todos los demás para que en semana 4 se pueda reproducir cualquier resultado.


## 4. Relación con CENTINELA+

MLflow va a producción cuando se desplieguen modelos reales. Versionar datasets evita el clásico 'funcionaba en mi máquina'.


## 5. Relación con Medallion

MLOps es **transversal**: versiona artefactos derivados de plata y oro.


## 6. Datos de entrada

Conceptual.


## 7. Schema CAPTIA esperado

No aplica.


## 8. Setup y variables de entorno

Cargamos las variables de entorno (`.env`), inicializamos `numpy` con `seed=42` y aplicamos el estilo de plotting compartido. Los helpers viven en `notebooks/_common/`.


In [ ]:
# Setup canónico — todos los notebooks didácticos lo usan
from __future__ import annotations

import sys
from pathlib import Path

ROOT = Path.cwd()
while ROOT.name and not (ROOT / "pyproject.toml").exists():
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from notebooks._common.captia_schema import (
    CANONICAL_TAGS, MEASUREMENT_TELEMETRY, MEASUREMENT_FAULT_LABELS,
    DEFAULT_BUCKET_RETENTIONS, KNOWN_VARIABLES,
    build_topic, build_line_protocol, validate_canonical_tags,
)
from notebooks._common.connection import load_env, get_influx_client, get_default_bucket
from notebooks._common.plotting import setup_default_style, plot_timeseries, plot_distribution
import notebooks._common.synthetic_mocks as mocks

SEED = 42
rng = np.random.default_rng(SEED)
setup_default_style()
load_env()
print(f"ROOT={ROOT}, SEED={SEED}, default_bucket={get_default_bucket()}")


## 9. Carga de datos o mock

Construimos un mapa conceptual.


In [ ]:
mlflow_concepts = pd.DataFrame(
    [
        ("Experiment", "Agrupación lógica", "case_B_baseline_2026"),
        ("Run", "Ejecución concreta", "rf_v3_seed42"),
        ("Param", "Hiperparámetro", "n_estimators=200"),
        ("Metric", "Indicador", "MAE=12.4"),
        ("Artifact", "Modelo / plot / dataset", "model.pkl, residuos.png"),
        ("Tag", "Metadata libre", "stage=staging"),
    ],
    columns=["objeto", "papel", "ejemplo"],
)
mlflow_concepts


## 10. Exploración paso a paso

lakeFS funciona como Git pero para datos: branches, commits y tags.


In [ ]:
lakefs_concepts = pd.DataFrame(
    [
        ("Repository", "Bucket lógico (S3 backend)", "captia-datasets"),
        ("Branch", "Línea de trabajo", "main / experiment-Y"),
        ("Commit", "Snapshot inmutable", "deadbeef"),
        ("Tag", "Etiqueta humana", "case_B/baseline_v1"),
        ("Hooks", "Validaciones pre/post commit", "schema check"),
    ],
    columns=["objeto", "papel", "ejemplo"],
)
lakefs_concepts


## 11. Transformación bronce → plata

No aplica.


## 12. Construcción de capa oro

MLflow + lakeFS *son* la capa oro de F.


## 13. Visualizaciones explicativas

Diagrama relacional.


In [ ]:
from IPython.display import Markdown
Markdown('''```mermaid
flowchart LR
  D[(lakeFS\nDataset versionado)] --> R[MLflow Run]
  R --> A[Artefactos\nmodel.pkl + plots]
  R --> M[Métricas\nMAE/MAPE/RMSE]
  R --> T[Tag\nlakeFS_tag=...]
  T --> D
```''')


## 14. Validaciones

El conocimiento, no datos.


## 15. Errores comunes

1. Crear un run nuevo cada vez que cambias un hiperparámetro pero no marcar la experiment.
2. No registrar el lakeFS tag en el run.
3. Subir artefactos enormes (CSVs) en lugar de versionarlos en lakeFS.


## 16. Ejercicios propuestos

1. Diseña la convención de naming para tu equipo.
2. Escribe la regla `pre_commit` que valida el schema CAPTIA en lakeFS.
3. Discute cuándo subir un artefacto vs cuándo versionar el dataset.


## 17. Cómo se reutiliza con datos reales

El stack MLflow + lakeFS escala a producción sin cambios.


## 18. Resumen final y próximos pasos

Recuerda los conceptos principales del notebook y enlaza al siguiente paso.

- Siguiente notebook: `06_case_F_mlops/02_tracking_experimentos.ipynb`.
- Documento web del caso: `docs/use-cases/case-f-mlops.md`.
